# Extinction JWST 
---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from matplotlib import image
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
import numpy as np
from collections import Counter
from scipy.spatial import cKDTree as KDT
from astropy.table import Column, Table
import itertools
import copy
import scipy.signal
from scipy.spatial import distance
import math
import sys
import pdb
import pickle as pickle
from astropy.convolution import convolve, Gaussian2DKernel
import astropy.coordinates as coord
import astropy.units as u
from astropy.io import ascii
import mpl_scatter_density
from scipy.stats import gaussian_kde, kde
from spisea import synthetic, evolution, atmospheres, reddening, ifmr
from spisea.imf import imf, multiplicity
import pdb
import pylab as py
from flystar import match, transforms, plots, align
from flystar import starlists
from flystar.starlists import StarList
from flystar.startables import StarTable
from astropy.table import Table, Column, vstack
import flystar
import datetime
import copy
import os
import pdb
import time
import warnings
from astropy.utils.exceptions import AstropyUserWarning

### Generating CMDs
---

In [ ]:
def matched(catalog1, catalog2, catalog1_name, catalog2_name, dr_tol, dm_tol):
    
    """ 
    catalog1: Pandas DataFrame
    catalog2: Pandas DataFrame
    dr_tol: radius tolerance
    dm_tol: magnitude tolerance
    """
    
    df = catalog1
    df2 = catalog2

    x1 = df['x']        # x centroid position of catalog1 stars
    y1 = df['y']        # y centroid position of catalog1 stars
    m1 = df['m']        # vega magnitude of catalog1 stars
    me1 = df['me']      # e rror in vega magnitude of catalog1 stars
    x2 = df2['x']
    y2 = df2['y']
    m2 = df2['m']
    me2 = df2['me']

    idxs1, idxs2, dr, dm = flystar.match.match(x1, y1, m1, 
                                               x2, y2, m2,
                                               dr_tol, dm_tol,
                                               verbose = False
                                              )
    m1_error = []
    m2_error = []
    m1_matched = []
    m2_matched = []
    m_difference = []

    for i in idxs1:
        m1_error += [me1[i]]
        m1_matched += [m1[i]]
    for i in idxs2:
        m2_error += [me2[i]]
        m2_matched += [m2[i]]

    m_difference = np.subtract(m1_matched, m2_matched)
    m1_error = np.array(m1_error)
    m2_error = np.array(m2_error)

    print(f" A total of {len(m1_matched)} stars matched between {catalog1_name} and {catalog2_name}")

    return m1_matched, m2_matched, m1_error, m2_error, m_difference

In [ ]:
N1_115_catalog = Table.read("catalogs/dr2/NRCB1_catalog115w.csv").to_pandas()
N1_212_catalog = Table.read("catalogs/dr2/NRCB1_catalog212n.csv").to_pandas()
N1_115_matched, N1_212_matched, N1_115_error, N1_212_error, N1_115_212_difference = matched(N1_115_catalog, N1_212_catalog, "NRCB1_F115W", "NRCB1_F212N", 0.5, 99)

N2_115_catalog = Table.read("catalogs/dr2/NRCB2_catalog115w.csv").to_pandas()
N2_212_catalog = Table.read("catalogs/dr2/NRCB2_catalog212n.csv").to_pandas()
N2_115_matched, N2_212_matched, N2_115_error, N2_212_error, N2_115_212_difference = matched(N2_115_catalog, N2_212_catalog, "NRCB2_F115W", "NRCB2_F212N", 0.5, 99)

N3_115_catalog = Table.read("catalogs/dr2/NRCB3_catalog115w.csv").to_pandas()
N3_212_catalog = Table.read("catalogs/dr2/NRCB3_catalog212n.csv").to_pandas()
N3_115_matched, N3_212_matched, N3_115_error, N3_212_error, N3_115_212_difference = matched(N3_115_catalog, N3_212_catalog, "NRCB3_F115W", "NRCB3_F212N", 0.5, 99)

N4_115_catalog = Table.read("catalogs/dr2/NRCB4_catalog115w.csv").to_pandas()
N4_212_catalog = Table.read("catalogs/dr2/NRCB4_catalog212n.csv").to_pandas()
N4_115_matched, N4_212_matched, N4_115_error, N4_212_error, N4_115_212_difference = matched(N4_115_catalog, N4_212_catalog, "NRCB4_F115W", "NRCB4_F212N", 0.5, 99)

fig, axis = plt.subplots(4, 2, figsize = (20, 20))
fig.suptitle(' NRCB1-NRCB4 All Color-Magnitude Diagrams ', fontsize=15)

axis[0,0].invert_yaxis()
axis[0,1].invert_yaxis()
axis[1,0].invert_yaxis()
axis[1,1].invert_yaxis()
axis[2,0].invert_yaxis()
axis[2,1].invert_yaxis()
axis[3,0].invert_yaxis()
axis[3,1].invert_yaxis()

xy = np.vstack([N1_115_212_difference, N1_115_matched])
z = gaussian_kde(xy)(xy)
axis[0,0].scatter(N1_115_212_difference, N1_115_matched, c = z, s = 1)
axis[0,0].set_xlabel('NRCB1 F115W - F212N')
axis[0,0].set_ylabel('NRCB1 F115W')
axis[0,0].set_title('NRCB1 F115W vs. F115W - F212N')
xy = np.vstack([N1_115_212_difference, N1_212_matched])
z = gaussian_kde(xy)(xy)
axis[0,1].scatter(N1_115_212_difference, N1_212_matched, c = z, s = 1)
axis[0,1].set_xlabel('NRCB1 F115W - F212N')
axis[0,1].set_ylabel('NRCB1 F212N')
axis[0,1].set_title('NRCB1 F212N vs. F115W - F212N')

xy = np.vstack([N2_115_212_difference, N2_115_matched])
z = gaussian_kde(xy)(xy)
axis[1,0].scatter(N2_115_212_difference, N2_115_matched, c = z, s = 1)
axis[1,0].set_xlabel('NRCB2 F115W - F212N')
axis[1,0].set_ylabel('NRCB2 F115W')
axis[1,0].set_title('NRCB2 F115W vs. F115W - F212N')
xy = np.vstack([N2_115_212_difference, N2_212_matched])
z = gaussian_kde(xy)(xy)
axis[1,1].scatter(N2_115_212_difference, N2_212_matched, c = z, s = 1)
axis[1,1].set_xlabel('NRCB2 F115W - F212N')
axis[1,1].set_ylabel('NRCB2 F212N')
axis[1,1].set_title('NRCB2 F212N vs. F115W - F212N')

xy = np.vstack([N3_115_212_difference, N3_115_matched])
z = gaussian_kde(xy)(xy)
axis[2,0].scatter(N3_115_212_difference, N3_115_matched, c = z, s = 1)
axis[2,0].set_xlabel('NRCB3 F115W - F212N')
axis[2,0].set_ylabel('NRCB3 F115W')
axis[2,0].set_title('NRCB3 F115W vs. F115W - F212N')
xy = np.vstack([N3_115_212_difference, N3_212_matched])
z = gaussian_kde(xy)(xy)
axis[2,1].scatter(N3_115_212_difference, N3_212_matched, c = z, s = 1)
axis[2,1].set_title('NRCB3 F212N vs. F115W - F212N')
axis[2,1].set_xlabel('NRCB3 F115W - F212N')
axis[2,1].set_ylabel('NRCB3 F212N')

xy = np.vstack([N4_115_212_difference, N4_115_matched])
z = gaussian_kde(xy)(xy)
axis[3,0].scatter(N4_115_212_difference, N4_115_matched, c = z, s = 1)
axis[3,0].set_xlabel('NRCB4 F115W - F212N')
axis[3,0].set_ylabel('NRCB4 F115W')
axis[3,0].set_title('NRCB4 F115W vs. F115W - F212N')
xy = np.vstack([N4_115_212_difference, N4_212_matched])
z = gaussian_kde(xy)(xy)
axis[3,1].scatter(N4_115_212_difference, N4_212_matched, c = z, s = 1)
axis[3,1].set_xlabel('NRCB4 F115W - F212N')
axis[3,1].set_ylabel('NRCB4 F212N')
axis[3,1].set_title('NRCB4 F212N vs. F115W - F212N')
fig.tight_layout()
file_name = "NRCB1_NRCB4_CMDs"
plt.savefig(f"/Users/devaldeliwala/research/jwst/cmds/dr2/{file_name}")

### Generating Error v. Magnitude Diagrams
---



In [ ]:
fig, axis = plt.subplots(4, 2, figsize = (20, 20))

fig.suptitle(' NRCB1-NRCB4 All Error-Magnitude Diagrams ', fontsize=15)

axis[0,0].scatter(N1_115_matched, N1_115_error, s = 1, marker = '+', c = 'black')
axis[0,1].scatter(N1_212_matched, N1_212_error, s = 1, marker = '+', c = 'black')
axis[1,0].scatter(N2_115_matched, N2_115_error, s = 1, marker = '+', c = 'black')
axis[1,1].scatter(N2_212_matched, N2_212_error, s = 1, marker = '+', c = 'black')
axis[2,0].scatter(N3_115_matched, N3_115_error, s = 1, marker = '+', c = 'black')
axis[2,1].scatter(N3_212_matched, N3_212_error, s = 1, marker = '+', c = 'black')
axis[3,0].scatter(N4_115_matched, N4_115_error, s = 1, marker = '+', c = 'black')
axis[3,1].scatter(N4_212_matched, N4_212_error, s = 1, marker = '+', c = 'black')

axis[0,0].set_xlabel("NRCB1_F115W mag")
axis[0,0].set_ylabel("NRCB1_F115W magσ ")
axis[0,0].set_title("NRCB1 F115W Error-Mag")
axis[0,1].set_xlabel("NRCB1_F212N mag")
axis[0,1].set_ylabel("NRCB1_F212N magσ ")
axis[0,1].set_title("NRCB1 F212N Error-Mag")

axis[1,0].set_xlabel("NRCB2_F115W mag")
axis[1,0].set_ylabel("NRCB2_F115W magσ ")
axis[1,0].set_title("NRCB2 F115W Error_Mag")
axis[1,1].set_xlabel("NRCB2_F212N mag")
axis[1,1].set_ylabel("NRCB2_F212N magσ ")
axis[1,1].set_title("NRCB2 F212N Error-Mag")

axis[2,0].set_xlabel("NRCB3_F115W mag")
axis[2,0].set_ylabel("NRCB3_F115W magσ ")
axis[2,0].set_title("NRCB3 F115W Error_Mag")
axis[2,1].set_xlabel("NRCB3_F212N mag")
axis[2,1].set_ylabel("NRCB3_F212N magσ ")
axis[2,1].set_title("NRCB3 F212N Error-Mag")

axis[3,0].set_xlabel("NRCB4_F115W mag")
axis[3,0].set_ylabel("NRCB4_F115W magσ ")
axis[3,0].set_title("NRCB4 F115W Error_Mag")
axis[3,1].set_xlabel("NRCB4_F212N mag")
axis[3,1].set_ylabel("NRCB4_F212N magσ ")
axis[3,1].set_title("NRCB4 F212N Error-Mag")

fig.tight_layout()
file_name = "NRCB1-NRCB4_ErrorMag"
plt.savefig(f"/Users/devaldeliwala/research/jwst/cmds/dr2/{file_name}.png")

### Generating Isochrones
---

In [ ]:
def generate_isochrones(catalog1_name, catalog2_name,
                        logAge, AKs, AKs_step, dist, metallicity, mass, iso_dir
):
    
    evo_model = evolution.MISTv1()
    atm_func  = atmospheres.get_merged_atmosphere
    red_law   = reddening.RedLawFritz11(scale_lambda=2.166)
    filt_list = [catalog1_name, catalog2_name]

    AKs2 = AKs  + AKs_step
    AKs3 = AKs2 + AKs_step
    AKs4 = AKs3 + AKs_step
    AKs5 = AKs4 + AKs_step

    my_iso = synthetic.IsochronePhot(logAge, AKs, dist,
                                        metallicity,
                                        evo_model=evo_model, atm_func=atm_func,
                                        red_law=red_law, filters=filt_list,
                                        iso_dir=iso_dir)

    my_iso2 = synthetic.IsochronePhot(logAge, AKs2, dist, metallicity,
                                        evo_model=evo_model, atm_func=atm_func,
                                        red_law=red_law, filters=filt_list,
                                        iso_dir=iso_dir)

    my_iso3 = synthetic.IsochronePhot(logAge, AKs3, dist, metallicity,
                                        evo_model=evo_model, atm_func=atm_func,
                                        red_law=red_law, filters=filt_list,
                                        iso_dir=iso_dir)

    my_iso4 = synthetic.IsochronePhot(logAge, AKs4, dist, metallicity,
                                        evo_model=evo_model, atm_func=atm_func,
                                        red_law=red_law, filters=filt_list,
                                        iso_dir=iso_dir)

    my_iso5 = synthetic.IsochronePhot(logAge, AKs5, dist, metallicity,
                                        evo_model=evo_model, atm_func=atm_func,
                                        red_law=red_law, filters=filt_list,
                                        iso_dir=iso_dir)

    file_name = filt_list[0] + "_" + filt_list[1]

    dfy = pd.DataFrame(my_iso.points['phase'])
    dfy.to_csv(f"{iso_dir}" + f"spisea_iso{file_name}.csv")

    print('The columns in the isochrone table are:\
            {0}'.format(my_iso.points.keys()))


    idx = np.where( abs(my_iso.points['mass'] - 1.0) == min(abs(my_iso.points['mass'] - 1.0)) )[0]
    filter_1 = np.round(my_iso.points[idx[0]][''+my_iso.points.keys()[9]], decimals=3)
    filter_2 = np.round(my_iso.points[idx[0]][''+my_iso.points.keys()[8]], decimals=3)

    idx2 = np.where( abs(my_iso2.points['mass'] - 1.0) == min(abs(my_iso2.points['mass'] - 1.0)) )[0]
    filter1_2 = np.round(my_iso2.points[idx2[0]][''+my_iso2.points.keys()[9]], decimals=3)
    filter2_2 = np.round(my_iso2.points[idx2[0]][''+my_iso2.points.keys()[8]], decimals=3)

    idx3 = np.where( abs(my_iso3.points['mass'] - 1.0) == min(abs(my_iso3.points['mass'] - 1.0)) )[0]
    filter1_3 = np.round(my_iso3.points[idx3[0]][''+my_iso3.points.keys()[9]], decimals=3)
    filter2_3 = np.round(my_iso3.points[idx3[0]][''+my_iso3.points.keys()[8]], decimals=3)

    idx4 = np.where( abs(my_iso4.points['mass'] - 1.0) == min(abs(my_iso4.points['mass'] - 1.0)) )[0]
    filter1_4 = np.round(my_iso4.points[idx4[0]][''+my_iso4.points.keys()[9]], decimals=3)
    filter2_4 = np.round(my_iso4.points[idx4[0]][''+my_iso4.points.keys()[8]], decimals=3)

    idx5 = np.where( abs(my_iso5.points['mass'] - 1.0) == min(abs(my_iso5.points['mass'] - 1.0)) )[0]
    filter1_5 = np.round(my_iso5.points[idx5[0]][''+my_iso5.points.keys()[9]], decimals=3)
    filter2_5 = np.round(my_iso5.points[idx5[0]][''+my_iso5.points.keys()[8]], decimals=3)

    print('1 M_sun: {0} = {1} mag, {2} = {3}\
            mag'.format(catalog1_name, filter_1, catalog2_name,
            filter_2))


    return my_iso, my_iso2, my_iso3, my_iso4, my_iso5, idx, idx2, idx3, idx4, idx5



In [ ]:
fig, axis = plt.subplots(4, 2, figsize = (20, 20))
fig.suptitle(' NRCB1-NRCB4 All Color-Magnitude Diagrams ', fontsize=15)

axis[0,0].invert_yaxis()
axis[0,1].invert_yaxis()
axis[1,0].invert_yaxis()
axis[1,1].invert_yaxis()
axis[2,0].invert_yaxis()
axis[2,1].invert_yaxis()
axis[3,0].invert_yaxis()
axis[3,1].invert_yaxis()

my_iso, my_iso2, my_iso3, my_iso4, my_iso5, idx, idx2, idx3, idx4, idx5 = generate_isochrones(catalog1_name = 'jwst,F115W', catalog2_name = 'jwst,F212N',  logAge = np.log(10**9), AKs = 2, AKs_step = 0.5, dist = 8000, metallicity = -0.3, mass = 10**5, iso_dir = "/Users/devaldeliwala/research/jwst/isochrones/")

xy = np.vstack([N1_115_212_difference, N1_115_matched])
z = gaussian_kde(xy)(xy)
axis[0,0].scatter(N1_115_212_difference, N1_115_matched, c = z, s = 1)
axis[0,0].set_xlabel('NRCB1 F115W - F212N')
axis[0,0].set_ylabel('NRCB1 F115W')
axis[0,0].set_title('NRCB1 F115W vs. F115W - F212N')

axis[0,0].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[8]], 'r-', label='_nolegend_')
axis[0,0].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[8]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[0,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[8]], 'r-', label='_nolegend_')
axis[0,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[8]][idx2], 'b*', ms=15, label='_nolegend_')

axis[0,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[8]], 'r-', label='_nolegend_')
axis[0,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[8]][idx3], 'b*', ms=15, label='_nolegend_')

axis[0,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[8]], 'r-', label='_nolegend_')
axis[0,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[8]][idx4], 'b*', ms=15, label='_nolegend_')

axis[0,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[8]], 'r-', label='_nolegend_')
axis[0,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[8]][idx5], 'b*', ms=15, label='_nolegend_')

axis[0,1].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[9]], 'r-', label='_nolegend_')
axis[0,1].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[9]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[0,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[9]], 'r-', label='_nolegend_')
axis[0,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[9]][idx2], 'b*', ms=15, label='_nolegend_')

axis[0,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[9]], 'r-', label='_nolegend_')
axis[0,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[9]][idx3], 'b*', ms=15, label='_nolegend_')

axis[0,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[9]], 'r-', label='_nolegend_')
axis[0,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[9]][idx4], 'b*', ms=15, label='_nolegend_')

axis[0,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[9]], 'r-', label='_nolegend_')
axis[0,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[9]][idx5], 'b*', ms=15, label='_nolegend_')

xy = np.vstack([N1_115_212_difference, N1_212_matched])
z = gaussian_kde(xy)(xy)
axis[0,1].scatter(N1_115_212_difference, N1_212_matched, c = z, s = 1)
axis[0,1].set_xlabel('NRCB1 F115W - F212N')
axis[0,1].set_ylabel('NRCB1 F212N')
axis[0,1].set_title('NRCB1 F212N vs. F115W - F212N')

xy = np.vstack([N2_115_212_difference, N2_115_matched])
z = gaussian_kde(xy)(xy)
axis[1,0].scatter(N2_115_212_difference, N2_115_matched, c = z, s = 1)
axis[1,0].set_xlabel('NRCB2 F115W - F212N')
axis[1,0].set_ylabel('NRCB2 F115W')
axis[1,0].set_title('NRCB2 F115W vs. F115W - F212N')

axis[1,0].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[8]], 'r-', label='_nolegend_')
axis[1,0].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[8]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[1,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[8]], 'r-', label='_nolegend_')
axis[1,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[8]][idx2], 'b*', ms=15, label='_nolegend_')

axis[1,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[8]], 'r-', label='_nolegend_')
axis[1,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[8]][idx3], 'b*', ms=15, label='_nolegend_')

axis[1,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[8]], 'r-', label='_nolegend_')
axis[1,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[8]][idx4], 'b*', ms=15, label='_nolegend_')

axis[1,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[8]], 'r-', label='_nolegend_')
axis[1,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[8]][idx5], 'b*', ms=15, label='_nolegend_')

axis[1,1].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[9]], 'r-', label='_nolegend_')
axis[1,1].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[9]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[1,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[9]], 'r-', label='_nolegend_')
axis[1,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[9]][idx2], 'b*', ms=15, label='_nolegend_')

axis[1,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[9]], 'r-', label='_nolegend_')
axis[1,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[9]][idx3], 'b*', ms=15, label='_nolegend_')

axis[1,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[9]], 'r-', label='_nolegend_')
axis[1,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[9]][idx4], 'b*', ms=15, label='_nolegend_')

axis[1,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[9]], 'r-', label='_nolegend_')
axis[1,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[9]][idx5], 'b*', ms=15, label='_nolegend_')

xy = np.vstack([N2_115_212_difference, N2_212_matched])
z = gaussian_kde(xy)(xy)
axis[1,1].scatter(N2_115_212_difference, N2_212_matched, c = z, s = 1)
axis[1,1].set_xlabel('NRCB2 F115W - F212N')
axis[1,1].set_ylabel('NRCB2 F212N')
axis[1,1].set_title('NRCB2 F212N vs. F115W - F212N')

xy = np.vstack([N3_115_212_difference, N3_115_matched])
z = gaussian_kde(xy)(xy)
axis[2,0].scatter(N3_115_212_difference, N3_115_matched, c = z, s = 1)
axis[2,0].set_xlabel('NRCB3 F115W - F212N')
axis[2,0].set_ylabel('NRCB3 F115W')
axis[2,0].set_title('NRCB3 F115W vs. F115W - F212N')

axis[2,0].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[8]], 'r-', label='_nolegend_')
axis[2,0].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[8]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[2,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[8]], 'r-', label='_nolegend_')
axis[2,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[8]][idx2], 'b*', ms=15, label='_nolegend_')

axis[2,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[8]], 'r-', label='_nolegend_')
axis[2,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[8]][idx3], 'b*', ms=15, label='_nolegend_')

axis[2,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[8]], 'r-', label='_nolegend_')
axis[2,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[8]][idx4], 'b*', ms=15, label='_nolegend_')

axis[2,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[8]], 'r-', label='_nolegend_')
axis[2,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[8]][idx5], 'b*', ms=15, label='_nolegend_')

axis[2,1].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[9]], 'r-', label='_nolegend_')
axis[2,1].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[9]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[2,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[9]], 'r-', label='_nolegend_')
axis[2,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[9]][idx2], 'b*', ms=15, label='_nolegend_')

axis[2,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[9]], 'r-', label='_nolegend_')
axis[2,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[9]][idx3], 'b*', ms=15, label='_nolegend_')

axis[2,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[9]], 'r-', label='_nolegend_')
axis[2,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[9]][idx4], 'b*', ms=15, label='_nolegend_')

axis[2,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
    - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[9]], 'r-', label='_nolegend_')
axis[2,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[9]][idx5], 'b*', ms=15, label='_nolegend_')

xy = np.vstack([N3_115_212_difference, N3_212_matched])
z = gaussian_kde(xy)(xy)
axis[2,1].scatter(N3_115_212_difference, N3_212_matched, c = z, s = 1)
axis[2,1].set_title('NRCB3 F212N vs. F115W - F212N')
axis[2,1].set_xlabel('NRCB3 F115W - F212N')
axis[2,1].set_ylabel('NRCB3 F212N')

axis[3,0].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[8]], 'r-', label='_nolegend_')
axis[3,0].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[8]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[3,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[8]], 'r-', label='_nolegend_')
axis[3,0].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[8]][idx2], 'b*', ms=15, label='_nolegend_')

axis[3,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[8]], 'r-', label='_nolegend_')
axis[3,0].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[8]][idx3], 'b*', ms=15, label='_nolegend_')

axis[3,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[8]], 'r-', label='_nolegend_')
axis[3,0].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[8]][idx4], 'b*', ms=15, label='_nolegend_')

axis[3,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[8]], 'r-', label='_nolegend_')
axis[3,0].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[8]][idx5], 'b*', ms=15, label='_nolegend_')

axis[3,1].plot(my_iso.points[''+my_iso.points.keys()[8]]
        - my_iso.points[''+my_iso.points.keys()[9]],
    my_iso.points[''+my_iso.points.keys()[9]], 'r-', label='_nolegend_')
axis[3,1].plot(my_iso.points[''+my_iso.points.keys()[8]][idx]
        - my_iso.points[''+my_iso.points.keys()[9]][idx],
    my_iso.points[''+my_iso.points.keys()[9]][idx], 'b*', ms=15, label='1 $M_\odot$')

axis[3,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]]
        - my_iso2.points[''+my_iso2.points.keys()[9]],
    my_iso2.points[''+my_iso2.points.keys()[9]], 'r-', label='_nolegend_')
axis[3,1].plot(my_iso2.points[''+my_iso2.points.keys()[8]][idx2]
        - my_iso2.points[''+my_iso2.points.keys()[9]][idx2],
    my_iso2.points[''+my_iso2.points.keys()[9]][idx2], 'b*', ms=15, label='_nolegend_')

axis[3,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]]
        - my_iso3.points[''+my_iso3.points.keys()[9]],
    my_iso3.points[''+my_iso3.points.keys()[9]], 'r-', label='_nolegend_')
axis[3,1].plot(my_iso3.points[''+my_iso3.points.keys()[8]][idx3]
        - my_iso3.points[''+my_iso3.points.keys()[9]][idx3],
    my_iso3.points[''+my_iso3.points.keys()[9]][idx3], 'b*', ms=15, label='_nolegend_')

axis[3,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]]
    - my_iso4.points[''+my_iso4.points.keys()[9]],
    my_iso4.points[''+my_iso4.points.keys()[9]], 'r-', label='_nolegend_')
axis[3,1].plot(my_iso4.points[''+my_iso4.points.keys()[8]][idx4]
    - my_iso4.points[''+my_iso4.points.keys()[9]][idx4],
    my_iso4.points[''+my_iso4.points.keys()[9]][idx4], 'b*', ms=15, label='_nolegend_')

axis[3,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]]
        - my_iso5.points[''+my_iso5.points.keys()[9]],
    my_iso5.points[''+my_iso5.points.keys()[9]], 'r-', label='_nolegend_')
axis[3,1].plot(my_iso5.points[''+my_iso5.points.keys()[8]][idx5]
        - my_iso5.points[''+my_iso5.points.keys()[9]][idx5],
    my_iso5.points[''+my_iso5.points.keys()[9]][idx5], 'b*', ms=15, label='_nolegend_')

xy = np.vstack([N4_115_212_difference, N4_115_matched])
z = gaussian_kde(xy)(xy)
axis[3,0].scatter(N4_115_212_difference, N4_115_matched, c = z, s = 1)
axis[3,0].set_xlabel('NRCB4 F115W - F212N')
axis[3,0].set_ylabel('NRCB4 F115W')
axis[3,0].set_title('NRCB4 F115W vs. F115W - F212N')
xy = np.vstack([N4_115_212_difference, N4_212_matched])
z = gaussian_kde(xy)(xy)
axis[3,1].scatter(N4_115_212_difference, N4_212_matched, c = z, s = 1)
axis[3,1].set_xlabel('NRCB4 F115W - F212N')
axis[3,1].set_ylabel('NRCB4 F212N')
axis[3,1].set_title('NRCB4 F212N vs. F115W - F212N')
fig.tight_layout()
file_name = "NRCB1-NRCB4_IsochroneCMD"
plt.savefig(f"/Users/devaldeliwala/research/jwst/isochrones/{file_name}.png")